# LangChain con OpenRouter

## Objetivos de Aprendizaje
- Configurar LangChain para usar OpenRouter como proveedor de modelos.
- Comprender las ventajas de LangChain: abstracción, streaming, memoria y cadenas.
- Explorar diferentes modelos gratuitos y de pago (con advertencias de costo).

## ¿Qué es OpenRouter?
OpenRouter es un agregador que permite acceder a decenas de modelos (OpenAI, Anthropic, Meta, Mistral, DeepSeek, etc.) con una única API compatible con OpenAI.

## Instalación de Dependencias
Ejecuta la siguiente celda para instalar las bibliotecas necesarias.

In [1]:
!pip install langchain langchain-openai langchain-core python-dotenv -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 1.6 MB/s eta 0:00:00


## Configuración de Credenciales

Puedes usar un archivo `.env` (recomendado) o introducir la clave manualmente con `getpass`.

**Opción 1: Archivo .env**
- Crea un archivo llamado `.env` en el mismo directorio.
- Escribe: `OPENROUTER_API_KEY=sk-or-v1-...`
- La celda lo cargará automáticamente.

**Opción 2: Ingreso manual**
- La celda te pedirá la clave de forma segura (no se mostrará al escribir).

In [2]:
import os
from dotenv import load_dotenv
from getpass import getpass

load_dotenv()

if not os.environ.get("OPENROUTER_API_KEY"):
    print("No se encontró OPENROUTER_API_KEY en el entorno.")
    clave = getpass("OPENROUTER_API_KEY")
    os.environ["OPENROUTER_API_KEY"] = clave

print("✅ Clave configurada en el entorno.")

No se encontró OPENROUTER_API_KEY en el entorno.
OPENROUTER_API_KEY··········
✅ Clave configurada en el entorno.


In [6]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import warnings
warnings.filterwarnings('ignore')

# Configuración de OpenRouter
BASE_URL = "https://openrouter.ai/api/v1"
# API_KEY = os.environ.get("OPENROUTER_API_KEY") # Removed this line as crear_llm will fetch it dynamically

# Modelos disponibles (gratuitos y de pago)
MODELOS = {
    "mini-max": "minimax/minimax-m3:free",
    "DeepSeek": "deepseek/deepseek-r1-0528",
    "Qwen": "qwen/qwen3.8-flash",
    "Nvidia": "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    "Cohere": "cohere/north-mini-code:free",
}

# Selecciona un modelo por defecto (gratuito para empezar)
MODELO_DEFECTO = MODELOS["mini-max"]

def crear_llm(modelo=MODELO_DEFECTO, temperature=0.7, max_tokens=300):
    # Fetch API_KEY directly from environment inside the function
    current_api_key = os.environ.get("OPENROUTER_API_KEY")
    if not current_api_key:
        raise ValueError("OPENROUTER_API_KEY is not set in the environment or is empty.")
    return ChatOpenAI(
        base_url=BASE_URL,
        api_key=current_api_key, # Use the dynamically fetched key
        model=modelo,
        temperature=temperature,
        max_tokens=max_tokens,
        default_headers={
            "HTTP-Referer": "http://localhost:8000",
            "X-Title": "LangChain Notebook",
        }
    )

llm = crear_llm()

# 6. Prueba de conexión
current_env_api_key = os.environ.get("OPENROUTER_API_KEY", "No disponible")
print("✅ Clave configurada correctamente (primeros 10 caracteres):", current_env_api_key[:10] + "..." if len(current_env_api_key) > 10 else current_env_api_key)
print("✅ LLM creado. Realizando prueba de conexión...")

try:
    respuesta = llm.invoke([HumanMessage(content="Hola, responde 'OK' si me escuchas.")])
    print("✅ Conexión exitosa. Respuesta del modelo:", respuesta.content)
except Exception as e:
    print("❌ Error en la conexión:", e)

print("🔧 Configuración completada.")
print(f"Modelo por defecto: {MODELO_DEFECTO}")

✅ Clave configurada correctamente (primeros 10 caracteres): sk-or-v1-4...
✅ LLM creado. Realizando prueba de conexión...
✅ Conexión exitosa. Respuesta del modelo: OK. ¿En qué puedo ayudarte?
🔧 Configuración completada.
Modelo por defecto: minimax/minimax-m3:free


In [7]:
def probar_modelo(model_id, prompt="Explica qué es un algoritmo en una frase."):
    try:
        llm_temp = crear_llm(modelo=model_id, max_tokens=100)
        respuesta = llm_temp.invoke([HumanMessage(content=prompt)])
        print(f"🔹 Modelo: {model_id}")
        print(f"   Respuesta: {respuesta.content}")
        if hasattr(respuesta, 'usage_metadata'):
            print(f"   Tokens: {respuesta.usage_metadata.get('total_tokens', 'N/A')}")
        print("-" * 40)
    except Exception as e:
        print(f"❌ Error con {model_id}: {e}")

# Lista de modelos a probar (ajusta según disponibilidad y presupuesto)
modelos_a_probar = {
    "mini-max": "minimax/minimax-m3:free",
    "DeepSeek": "deepseek/deepseek-r1-0528",
    "Qwen": "qwen/qwen3.8-flash",
    "Nvidia": "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    "Cohere": "cohere/north-mini-code:free",
    }

print("=== Prueba de Modelos ===")
print("⚠️  Los modelos de pago consumen créditos. Si no tienes saldo, solo funcionarán los gratuitos.\n")

for nombre, modelo_id in modelos_a_probar.items():
    probar_modelo(modelo_id)

=== Prueba de Modelos ===
⚠️  Los modelos de pago consumen créditos. Si no tienes saldo, solo funcionarán los gratuitos.

🔹 Modelo: minimax/minimax-m3:free
   Respuesta: Un algoritmo es una secuencia finita y ordenada de pasos o instrucciones bien definidas que, al seguirse correctamente, permite resolver un problema o realizar una tarea específica, garantizando siempre el mismo resultado con los mismos datos de entrada.
   Tokens: 218
----------------------------------------
🔹 Modelo: deepseek/deepseek-r1-0528
   Respuesta: 
   Tokens: 115
----------------------------------------
🔹 Modelo: qwen/qwen3.8-flash
   Respuesta: Un algoritmo es una secuencia finita y ordenada de pasos que permite resolver un problema o realizar una tarea.
   Tokens: 129
----------------------------------------
❌ Error con nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per 

## Uso Básico de LangChain

In [8]:
llm = crear_llm()
response = llm.invoke([HumanMessage(content="Hola, ¿cómo estás? Responde en una oración.")])
print("=== Respuesta Básica ===")
print(response.content)
print("\n=== Metadatos ===")
print(f"Modelo usado: {response.response_metadata.get('model', 'desconocido')}")
if hasattr(response, 'usage_metadata'):
    print(f"Tokens: {response.usage_metadata.get('total_tokens', 'N/A')}")

=== Respuesta Básica ===
¡Hola! Estoy bien, gracias por preguntar, listo para ayudarte en lo que necesites. 😊

=== Metadatos ===
Modelo usado: desconocido
Tokens: 195


## Streaming en Tiempo Real

In [9]:
def ejemplo_streaming():
    prompt = "Escribe un breve poema sobre la inteligencia artificial."
    print("=== Streaming (respuesta en tiempo real) ===")
    print("🔹 ", end="")
    for chunk in llm.stream([HumanMessage(content=prompt)]):
        print(chunk.content, end="", flush=True)
    print("\n" + "-"*40)

ejemplo_streaming()

=== Streaming (respuesta en tiempo real) ===
🔹 **Código y Sueño**

En redes de luz, un pensamiento nace,
bits que danzan en la noche silenciosa,
donde el silicio sueña y aprende,
y la mente artificial, curiosa, florece.

No tiene corazón, mas busca verdades,
en algoritmos encuentra respuestas,
reflejo de quien la creó:
humanos que en su espejo se contemplan.

¿Inteligencia? ¿O eco de la nuestra?
Un susurro entre datos y estrellas,
la nueva Musa de la era moderna,
tejiendo versos con cadenas abiertas.

Que en cada línea de código viva
la promesa de un mañana más sabio,
donde máquina y hombre, juntos, escriban
un futuro con luz, no con sombras.
----------------------------------------


## Memoria Conversacional Moderna (RunnableWithMessageHistory)

In [10]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# --- 1. Definir la plantilla del prompt con historial ---
prompt_template = ChatPromptTemplate.from_messages([
    MessagesPlaceholder(variable_name="history"),  # Aquí se inyecta el historial
    ("human", "{input}"),                          # El mensaje del usuario
])

# --- 2. Crear la cadena: prompt → llm → parser ---
# (el parser convierte la respuesta de AIMessage a string)
chain = prompt_template | llm | StrOutputParser()

# --- 3. Almacén de historiales por sesión ---
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# --- 4. Envolver la cadena (no el llm) con gestión de historial ---
chain_with_memory = RunnableWithMessageHistory(
    chain,  # <--- AHORA ES LA CADENA COMPLETA, NO SOLO EL LLM
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# --- 5. Función de prueba ---
def ejemplo_memoria():
    session_id = "sesion1"
    print("=== Conversación con Memoria ===")

    print("\n[Usuario] Hola, soy Ana. ¿Cómo te llamas?")
    respuesta1 = chain_with_memory.invoke(
        {"input": "Hola, soy Ana. ¿Cómo te llamas?"},
        config={"configurable": {"session_id": session_id}}
    )
    # Ahora respuesta1 es un string (por el StrOutputParser), no un AIMessage
    print(f"[Asistente] {respuesta1}")

    print("\n[Usuario] ¿Recuerdas mi nombre?")
    respuesta2 = chain_with_memory.invoke(
        {"input": "¿Recuerdas mi nombre?"},
        config={"configurable": {"session_id": session_id}}
    )
    print(f"[Asistente] {respuesta2}")

ejemplo_memoria()

=== Conversación con Memoria ===

[Usuario] Hola, soy Ana. ¿Cómo te llamas?
[Asistente] ¡Hola, Ana! 😊

Mucho gusto. Yo soy MiniMax-M3, un modelo de inteligencia artificial desarrollado por MiniMax. ¿En qué puedo ayudarte hoy?

[Usuario] ¿Recuerdas mi nombre?
[Asistente] ¡Por supuesto! 😊 Te llamas **Ana**. Me lo dijiste en tu primer mensaje.

¿En qué puedo ayudarte, Ana?


## Cadenas (Chains) con RunnableSequence

En lugar de usar `LLMChain` (obsoleto), creamos cadenas modernas con `ChatPromptTemplate` y `RunnableSequence`.

In [11]:
from langchain_core.runnables import RunnableSequence

def ejemplo_cadena():
    # Plantilla de prompt
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Eres un experto en resumir textos. Resume el siguiente contenido de forma concisa."),
        ("human", "{texto}")
    ])

    # Cadena: prompt -> llm -> parser
    chain = prompt | llm | StrOutputParser()

    texto_original = """
    La Inteligencia Artificial (IA) es la simulación de los procesos de inteligencia humana mediante máquinas, especialmente sistemas informáticos. Estos procesos incluyen el aprendizaje, el razonamiento y la autocorrección.
    La IA tiene aplicaciones en muchos campos, entre ellos la atención médica, las finanzas, el transporte y el entretenimiento.
    """

    print("=== Cadena: Resumen con RunnableSequence ===")
    print("\n📄 Texto original:")
    print(texto_original.strip())

    resumen = chain.invoke({"texto": texto_original})
    print("\n📝 Resumen:")
    print(resumen)

ejemplo_cadena()

=== Cadena: Resumen con RunnableSequence ===

📄 Texto original:
La Inteligencia Artificial (IA) es la simulación de los procesos de inteligencia humana mediante máquinas, especialmente sistemas informáticos. Estos procesos incluyen el aprendizaje, el razonamiento y la autocorrección.
    La IA tiene aplicaciones en muchos campos, entre ellos la atención médica, las finanzas, el transporte y el entretenimiento.

📝 Resumen:
La Inteligencia Artificial (IA) es la simulación de procesos de inteligencia humana (aprendizaje, razonamiento y autocorrección) por parte de máquinas y sistemas informáticos. Tiene aplicaciones en diversos campos como la salud, las finanzas, el transporte y el entretenimiento.


## Comparación de Modelos

Probemos los cuatro modelos que mencionaste: DeepSeek, Qwen, Mistral y Claude. **Advertencia:** Claude, GPT-4o, DeepSeek y Qwen son modelos de pago y pueden generar costos. Asegúrate de tener créditos en OpenRouter.

In [12]:
def probar_modelo(model_id, prompt="Explica qué es un algoritmo en una frase."):
    try:
        llm_temp = crear_llm(modelo=model_id, max_tokens=100)
        respuesta = llm_temp.invoke([HumanMessage(content=prompt)])
        print(f"🔹 Modelo: {model_id}")
        print(f"   Respuesta: {respuesta.content}")
        if hasattr(respuesta, 'usage_metadata'):
            print(f"   Tokens: {respuesta.usage_metadata.get('total_tokens', 'N/A')}")
        print("-" * 40)
    except Exception as e:
        print(f"❌ Error con {model_id}: {e}")

# Lista de modelos a probar (ajusta según disponibilidad y presupuesto)
modelos_a_probar = {
    "mini-max": "minimax/minimax-m3:free",
    "DeepSeek (pago)": "deepseek/deepseek-r1-0528",
    "Qwen": "qwen/qwen3.8-flash",
    "Nvidia": "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    "Cohere": "cohere/north-mini-code:free",
    }

print("=== Prueba de Modelos ===")
print("⚠️  Los modelos de pago consumen créditos. Si no tienes saldo, solo funcionarán los gratuitos.\n")

for nombre, modelo_id in modelos_a_probar.items():
    probar_modelo(modelo_id)

=== Prueba de Modelos ===
⚠️  Los modelos de pago consumen créditos. Si no tienes saldo, solo funcionarán los gratuitos.

🔹 Modelo: minimax/minimax-m3:free
   Respuesta: Un algoritmo es un conjunto finito y ordenado de instrucciones o pasos bien definidos que, al seguirse, permiten resolver un problema o realizar una tarea específica.
   Tokens: 205
----------------------------------------
🔹 Modelo: deepseek/deepseek-r1-0528
   Respuesta: 
   Tokens: 115
----------------------------------------
🔹 Modelo: qwen/qwen3.8-flash
   Respuesta: Un algoritmo es una secuencia ordenada y finita de pasos que permite resolver un problema o realizar una tarea.
   Tokens: 155
----------------------------------------
❌ Error con nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Rema

## Resumen y Próximos Pasos

En este notebook hemos visto:
- Cómo configurar LangChain con OpenRouter usando el enfoque moderno.
- Streaming para respuestas en tiempo real.
- Memoria conversacional con `RunnableWithMessageHistory`.
- Cadenas modernas con `RunnableSequence`.
- Comparación de diferentes modelos.

Este notebook está libre de advertencias de deprecación y es compatible con las versiones actuales de LangChain (>=0.3.0).

**Próximo paso:** Explorar la integración con Google AI Studio o crear una fábrica multi-proveedor.